In [ ]:
print("Запуск Варианта 1: Target Encoding химических фрагментов (fr_)...\n")

# 1. Восстанавливаем оригинальные названия признаков, чтобы найти колонки fr_
X_train_df = pd.DataFrame(X, columns=X_train.columns)
X_test_df = pd.DataFrame(X_test, columns=X_train.columns)

# Находим все признаки функциональных групп
fr_cols = [c for c in X_train_df.columns if c.startswith('fr_')]
print(f"Найдено {len(fr_cols)} признаков химических фрагментов для кодирования.")

# Целевая переменная для кодирования (возьмем логарифм IC50 как главный маркер активности)
target_for_encoding = np.log1p(y_ic50_raw)

# Создаем копии датасетов для перезаписи
X_train_encoded = X_train_df.copy()
X_test_encoded = X_test_df.copy()

# Применяем Target Encoding сглаженным методом (Smoothing), чтобы избежать переобучения
smoothing_weight = 10
global_mean = target_for_encoding.mean()

for col in fr_cols:
    # Считаем статистики по каждой группе фрагментов в трейне
    stats = pd.DataFrame({'feat_val': X_train_df[col], 'target': target_for_encoding})
    grouped = stats.groupby('feat_val')['target'].agg(['count', 'mean'])

    # Формула сглаженного кодирования (уберет шум для редких фрагментов)
    smooth_map = ((grouped['count'] * grouped['mean'] + smoothing_weight * global_mean) /
                  (grouped['count'] + smoothing_weight)).to_dict()

    # Перезаписываем признаки в Train и Test новыми "весами активности"
    X_train_encoded[col] = X_train_encoded[col].map(smooth_map).fillna(global_mean)
    X_test_encoded[col] = X_test_encoded[col].map(smooth_map).fillna(global_mean)

print("Target Encoding успешно завершен! Матрицы признаков переведены в шкалу активности.")

# Переводим в массивы numpy
X_enc = X_train_encoded.values
X_test_enc = X_test_encoded.values

# Обучение на закодированных признаках
y_ic50_trans = np.sqrt(y_ic50_raw)
y_cc50_trans = np.sqrt(y_cc50_raw)
y_si_trans   = np.log1p(y_si_raw)

# Параметры для IC50
cat_p_ic50 = {
    'iterations': 1500,
    'learning_rate': 0.04935,
    'depth': 6,
    'l2_leaf_reg': 8,
    'subsample': 0.6538,
    'min_data_in_leaf': 23,
    'random_seed': 42,
    'verbose': 100,
    'early_stopping_rounds': 200,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'use_best_model': True,
}

# Параметры для CC50
cat_p_cc50 = {
    'iterations': 3000,
    'learning_rate': 0.01621,
    'depth': 7,
    'l2_leaf_reg': 1,
    'subsample': 0.7138,
    'min_data_in_leaf': 12,
    'random_seed': 42,
    'verbose': 100,
    'early_stopping_rounds': 200,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'use_best_model': True,
}

# Параметры для SI
cat_p_si = {
    'iterations': 3000,
    'learning_rate': 0.01049,
    'depth': 7,
    'l2_leaf_reg': 8,
    'subsample': 0.8340,
    'min_data_in_leaf': 21,
    'random_seed': 42,
    'verbose': 100,
    'early_stopping_rounds': 200,
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'use_best_model': True,
}


N = 5
kf = KFold(n_splits=N, shuffle=True, random_state=42)

pred_ic50_trans = np.zeros(len(X_test))
pred_cc50_trans = np.zeros(len(X_test))
pred_si_trans   = np.zeros(len(X_test))

for fold, (tr, val) in enumerate(kf.split(X_enc), 1):
    Xtr, Xval = X_enc[tr], X_enc[val]

    m_ic50 = CatBoostRegressor(**cat_p_ic50)
    m_ic50.fit(Xtr, y_ic50_trans[tr], eval_set=(Xval, y_ic50_trans[val]), use_best_model=True)
    pred_ic50_trans += m_ic50.predict(X_test_enc) / N

    m_cc50 = CatBoostRegressor(**cat_p_cc50)
    m_cc50.fit(Xtr, y_cc50_trans[tr], eval_set=(Xval, y_cc50_trans[val]), use_best_model=True)
    pred_cc50_trans += m_cc50.predict(X_test_enc) / N

    m_si = CatBoostRegressor(**cat_p_si)
    m_si.fit(Xtr, y_si_trans[tr], eval_set=(Xval, y_si_trans[val]), use_best_model=True)
    pred_si_trans += m_si.predict(X_test_enc) / N

# Обратный перевод после честного усреднения
t_ic50 = np.clip(pred_ic50_trans, 0, None) ** 2
t_cc50 = np.clip(pred_cc50_trans, 0, None) ** 2
t_si   = np.expm1(np.clip(pred_si_trans, 0, None))

# Защитный клиппинг по перцентилям твоей команды
t_ic50 = np.clip(t_ic50, 0, ic50_p99)
t_cc50 = np.clip(t_cc50, 0, cc50_p99)
t_si   = np.clip(t_si,   0, si_p95)

submission_encoded = pd.DataFrame({
    "index": test["index"],
    "IC50": t_ic50,
    "CC50": t_cc50,
    "SI": t_si
})

submission_encoded.to_csv('target_encoded_fragments_submission.csv', index=False)
print("\nФайл 'target_encoded_fragments_submission.csv' успешно создан")